In [8]:
import pandas as pd
import sys, os
sys.path.append(os.path.abspath("../data_creation/plots_and_analysis"))
import config
from pathlib import Path
import numpy as np
from scipy.stats import mannwhitneyu
from itertools import combinations
# adjust paths in /data_creation/plots_and_analysis/config.py

In [9]:
metrics = ["RMSSE", "MASE", "sMAPE"]
div_order = [0, 5, 10]
pot_order = ["5000", "12000", "25000", "50000", "100000", "all", "source"]
runs = ["case1_run1", "case1_run2", "case1_run3", "case1_run4", "case1_run5"]

In [10]:
config.path_to_data

'/home/dev/projects/data'

Calculate statistical significance between OSD diversity levels for fixed pot target sizes.

In [11]:
for run in runs: 
    path_to_forecasts = f"{config.path_to_data}/{run}/outputs/forecasts/"
    eval_dir = Path(path_to_forecasts) / "evaluation"
    df = pd.read_csv(eval_dir / f"single_mets_{run}.csv")

    pot_size_results = []

    for div_step in sorted(df["div_step"].unique()):

        sub = df[df["div_step"] == div_step]
        
        available_sizes = sub["pot target size"].astype(str).unique()

        sizes = [
            size
            for size in pot_order
            if size in available_sizes
        ]


        for size1, size2 in combinations(sizes, 2):
            
            print(f"Comparing div_step {div_step}: {size1} vs {size2}")

            row = {
                "div_step": div_step,
                "pot_size_1": size1,
                "pot_size_2": size2,
            }

            for metric in metrics:

                x = sub.loc[
                    sub["pot target size"] == size1,
                    metric
                ].dropna()

                y = sub.loc[
                    sub["pot target size"] == size2,
                    metric
                ].dropna()

                _, p = mannwhitneyu(
                    x,
                    y,
                    alternative="two-sided"
                )

                row[f"{metric}_p"] = p

            pot_size_results.append(row)

    pot_size_pvalues = pd.DataFrame(pot_size_results)
    p_cols = [c for c in pot_size_pvalues.columns if c.endswith("_p")]
    pot_size_pvalues[p_cols] = pot_size_pvalues[p_cols].apply(
        lambda col: col.map(lambda x: f"{x:.2e}")
    )
    

    pot_size_pvalues["div_step"] = pd.Categorical(
        pot_size_pvalues["div_step"],
        categories=div_order,
        ordered=True
    )

    pot_size_pvalues["pot_size_1"] = pd.Categorical(
        pot_size_pvalues["pot_size_1"],
        categories=pot_order,
        ordered=True
    )

    pot_size_pvalues["pot_size_2"] = pd.Categorical(
        pot_size_pvalues["pot_size_2"],
        categories=pot_order,
        ordered=True
    )

    pot_size_pvalues = pot_size_pvalues.sort_values(
        ["div_step", "pot_size_1", "pot_size_2"]
    )

    pot_size_pvalues = pot_size_pvalues.reset_index(drop=True)

    pot_size_pvalues.to_csv(eval_dir / f"pot_size_pvalues_{run}.csv", index=False)

Comparing div_step 0: 5000 vs 12000
Comparing div_step 0: 5000 vs 25000
Comparing div_step 0: 5000 vs 50000
Comparing div_step 0: 5000 vs 100000
Comparing div_step 0: 5000 vs all
Comparing div_step 0: 5000 vs source
Comparing div_step 0: 12000 vs 25000
Comparing div_step 0: 12000 vs 50000
Comparing div_step 0: 12000 vs 100000
Comparing div_step 0: 12000 vs all
Comparing div_step 0: 12000 vs source
Comparing div_step 0: 25000 vs 50000
Comparing div_step 0: 25000 vs 100000
Comparing div_step 0: 25000 vs all
Comparing div_step 0: 25000 vs source
Comparing div_step 0: 50000 vs 100000
Comparing div_step 0: 50000 vs all
Comparing div_step 0: 50000 vs source
Comparing div_step 0: 100000 vs all
Comparing div_step 0: 100000 vs source
Comparing div_step 0: all vs source
Comparing div_step 5: 5000 vs 12000
Comparing div_step 5: 5000 vs 25000
Comparing div_step 5: 5000 vs 50000
Comparing div_step 5: 5000 vs 100000
Comparing div_step 5: 5000 vs all
Comparing div_step 5: 5000 vs source
Comparing div

In [12]:
pot_size_pvalues

,div_step,pot_size_1,pot_size_2,RMSSE_p,MASE_p,sMAPE_p
0,0,5000,12000,9.29e-15,8.17e-14,3.17e-01
1,0,5000,25000,3.58e-24,1.93e-23,7.21e-09
2,0,5000,50000,2.19e-25,6.19e-25,9.99e-07
3,0,5000,100000,3.02e-14,8.57e-12,7.14e-64
4,0,5000,all,3.58e-01,4.55e-01,1.59e-40
...,...,...,...,...,...,...
58,10,50000,all,6.62e-36,4.19e-32,1.10e-16
59,10,50000,source,6.15e-50,2.68e-45,1.72e-21
60,10,100000,all,2.71e-08,6.81e-09,2.42e-03
61,10,100000,source,1.08e-03,6.03e-04,8.24e-02


Calculate significance over OSDs for fixed potential target sizes

In [13]:
for run in runs:
    path_to_forecasts = f"{config.path_to_data}/{run}/outputs/forecasts/"
    eval_dir = Path(path_to_forecasts) / "evaluation"
    df = pd.read_csv(eval_dir / f"single_mets_{run}.csv")
    div_step_results = []

    for pot_size in pot_order:

        sub = df[df["pot target size"].astype(str) == pot_size]

        if sub.empty:
            continue

        div_steps = [
            d
            for d in div_order
            if d in sub["div_step"].unique()
        ]

        for div1, div2 in combinations(div_steps, 2):

            row = {
                "pot_target_size": pot_size,
                "div_step_1": div1,
                "div_step_2": div2,
            }

            for metric in metrics:

                x = sub.loc[
                    sub["div_step"] == div1,
                    metric
                ].dropna()

                y = sub.loc[
                    sub["div_step"] == div2,
                    metric
                ].dropna()

                _, p = mannwhitneyu(
                    x,
                    y,
                    alternative="two-sided"
                )

                row[f"{metric}_p"] = p

            div_step_results.append(row)

    div_step_pvalues = pd.DataFrame(div_step_results)

    p_cols = [c for c in div_step_pvalues.columns if c.endswith("_p")]
    div_step_pvalues[p_cols] = div_step_pvalues[p_cols].apply(
        lambda col: col.map(lambda x: f"{x:.2e}")
    )

    div_step_pvalues["pot_target_size"] = pd.Categorical(
        div_step_pvalues["pot_target_size"],
        categories=pot_order,
        ordered=True
    )

    div_step_pvalues["div_step_1"] = pd.Categorical(
        div_step_pvalues["div_step_1"],
        categories=div_order,
        ordered=True
    )

    div_step_pvalues["div_step_2"] = pd.Categorical(
        div_step_pvalues["div_step_2"],
        categories=div_order,
        ordered=True
    )

    div_step_pvalues = (
        div_step_pvalues
        .sort_values(
            ["pot_target_size", "div_step_1", "div_step_2"]
        )
        .reset_index(drop=True)
    )

    div_step_pvalues.to_csv(eval_dir / f"div_step_pvalues_{run}.csv", index=False)
div_step_pvalues

,pot_target_size,div_step_1,div_step_2,RMSSE_p,MASE_p,sMAPE_p
0,5000,0,5,3.51e-01,9.99e-01,9.47e-01
1,5000,0,10,4.15e-02,6.02e-01,7.09e-01
2,5000,5,10,2.61e-01,6.05e-01,6.66e-01
3,12000,0,5,3.71e-02,5.41e-01,2.65e-01
4,12000,0,10,2.57e-03,5.61e-01,7.22e-01
5,12000,5,10,3.79e-01,9.71e-01,1.47e-01
6,25000,0,5,1.65e-01,9.89e-01,3.12e-01
7,25000,0,10,1.36e-02,9.06e-01,4.72e-01
8,25000,5,10,2.80e-01,9.24e-01,7.82e-01
9,50000,0,5,5.13e-01,7.22e-01,9.11e-01
